# ⚖️ 01-数据结构与算法 · 主题 9：贪心

> 贪心 = **每一步都选当前看起来最优的**，并证明该策略不亏。
> 本 notebook 目标：**掌握贪心适用判断 → 背熟区间/跳跃/差分三类模板 → 吃透 8 道高频题 → 通过 25 道自测**。

**前置**：主题 8（DP，用于对比）。**运行环境**：Python 3.8+，无第三方依赖。

## §1 心智模型与适用判断

### 贪心三步

1. **分解**：把整体问题切成一步步的局部决策
2. **选策略**：每一步按某个「度量」选当前最优（最早结束/最大覆盖/最小成本）
3. **证明**：交换论证 / 归纳法说明局部最优不损害全局最优

### 贪心 vs DP（面试必对比）

| | 贪心 | DP |
|--|------|-----|
| 决策 | 每步只选一个 | 比较所有子方案 |
| 依赖 | 只依赖上一步 | 依赖多个子状态 |
| 证明 | 需要交换论证 | 转移方程天然正确 |
| 反例代价 | 一步错全错 | 无 |

**经典反例**：硬币找零 `[1,3,4]`，amount=6：贪心 4+1+1（3 枚）≠ 最优 3+3（2 枚）——贪心不成立时立刻转 DP。

### 高频贪心模式

- **区间类**：按右端点/左端点排序（调度、合并、气球）
- **可达类**：维护最远覆盖范围（跳跃游戏）
- **差分类**：相邻元素收益（股票、加油站）
- **分配类**：双指针从小匹配（饼干、孩子）

## §2 区间调度：无重叠区间（435）/ 最少箭（452）

### 435 无重叠区间

**题目**：移除最少区间使剩余互不重叠。

**贪心策略**：按**右端点升序**排序，保留「结束最早」的区间——给后面留最大空间：

$$\text{保留} = \max\{\text{不重叠区间数}\}, \quad \text{移除} = n - \text{保留}$$

**为什么按右端点**（交换论证核心）：任何最优解里，第一个区间若换成「结束最早的区间」不产生更多冲突 → 贪心选择可替换进最优解。

### 452 用最少数量的箭引爆气球

**等价转换**：重叠的气球可以一支箭一起射 → 求**最多不相交区间数**（按右端点排序，能合并的重叠合并）。

**复杂度**：$O(n \log n)$（排序主导），$O(1)$。

In [ ]:
def erase_overlap_intervals(intervals):
    # 移除最少 = n - 最多不重叠数；按右端点升序
    intervals.sort(key=lambda x: x[1])
    keep = 0
    end = float('-inf')
    for l, r in intervals:
        if l >= end:
            keep += 1
            end = r              # 更新当前保留区间的右端点
    return len(intervals) - keep

def find_min_arrow_shots(points):
    # 排序后统计「新箭」次数 = 不重叠区间数
    points.sort(key=lambda x: x[1])
    arrows = 0
    end = float('-inf')
    for l, r in points:
        if l > end:
            arrows += 1          # 新的一支箭
            end = r
    return arrows

print('erase_overlap_intervals:', erase_overlap_intervals([[1, 2], [2, 3], [3, 4], [1, 3]]))  # 1
print('erase_overlap_intervals:', erase_overlap_intervals([[1, 2], [1, 2], [1, 2]]))          # 2
print('min_arrow_shots:', find_min_arrow_shots([[10, 16], [2, 8], [1, 6], [7, 12]]))           # 2
print('min_arrow_shots:', find_min_arrow_shots([[1, 2], [3, 4], [5, 6]]))                      # 3

## §3 跳跃游戏（55 / 45）：最远可达范围

### 55 判断能否到达终点

**贪心**：维护当前能到达的最远下标：

$$\text{reach} = \max(\text{reach}, i + nums[i]), \quad \text{若 } i > \text{reach} \text{ 则失败}$$

**复杂度**：$O(n)$ 时间，$O(1)$ 空间。

### 45 最少跳跃次数

**贪心**：在「当前步可达区间」内挑能扩展最远的点作为下一步：

$$\text{step} \gets \text{step} + 1, \quad \text{当 } i \text{ 触及当前步右界时更新为区间内最大 } i + nums[i]$$

**复杂度**：$O(n)$，$O(1)$。

In [ ]:
def can_jump(nums):
    reach = 0
    for i, x in enumerate(nums):
        if i > reach:
            return False          # 当前位置已不可达
        reach = max(reach, i + x)
    return True

def jump(nums):
    # 最少跳跃次数：每次在可达区间内挑扩展最远的
    n = len(nums)
    steps = 0
    cur_end = 0      # 当前这一步能到达的右边界
    farthest = 0     # 已扫描区间内的最远可达
    for i in range(n - 1):
        farthest = max(farthest, i + nums[i])
        if i == cur_end:      # 到达当前步边界：必须再跳一次
            steps += 1
            cur_end = farthest
    return steps

print('can_jump([2,3,1,1,4]) =', can_jump([2, 3, 1, 1, 4]))    # True
print('can_jump([3,2,1,0,4]) =', can_jump([3, 2, 1, 0, 4]))    # False
print('jump([2,3,1,1,4]) =', jump([2, 3, 1, 1, 4]))            # 2
print('jump([2,3,0,1,4]) =', jump([2, 3, 0, 1, 4]))            # 2
print('jump([0]) =', jump([0]))                                # 0

## §3.1 跳跃游戏贪心正确性证明（面试白板）

**命题**：55 的 `reach` 贪心策略能判定「是否能到终点」。

**证明（归纳）**：令 $\text{reach}_i = \max_{j \le i}(j + nums[j])$ 表示扫描到 $i$ 时可达的最远下标。

- **基例**：$i=0$ 时 $\text{reach}_0 = nums[0] \ge 0$，位置 0 本身可达 ✓
- **归纳**：若前 $i$ 个位置都可达且 $i \le \text{reach}_{i-1}$，则位置 $i$ 可达；又因为 $i$ 可达，从 $i$ 能跳到的所有位置 $[i, i+nums[i]]$ 也都可达 → $\text{reach}_i \ge \max(\text{reach}_{i-1}, i+nums[i])$ 覆盖了全部可达位置 ✓
- **结论**：若某一步 $i > \text{reach}$，说明存在不可达的断层 → 返回 False；否则扫描结束 $\text{reach} \ge n-1$ 即终点可达 ✓

> 💡 **直觉**：贪心只需维护「最远可达前缀」，任何可行路径都必须落在该前缀内——所以"最远扩展"不会漏掉任何解（这就是 45 最少步数的贪心依据）。

## §4 股票 / 加油站：差分收益

### 122 买卖股票的最佳时机 II（无限次）

**贪心公式**：只累加**正收益差**：

$$\text{profit} = \sum_i \max(0,\ prices[i+1] - prices[i])$$

**为什么对**：任意交易序列的总收益等于若干相邻差的组合；每天「上涨就吃、下跌就躲」不会亏。

### 134 加油站

**核心公式**：

$$\text{剩余}(i) = gas[i] - cost[i], \quad \text{总剩余} < 0 \implies \text{无解}$$

**贪心结论**：从「累计剩余首次跌到负数」的下一站重新开始——若有解，起点必在该站之后：

$$\text{start} = i + 1 \quad \text{当 } \text{cur\_sum} < 0$$

**复杂度**：$O(n)$，$O(1)$。

In [ ]:
def max_profit_ii(prices):
    # 累加所有上涨段
    profit = 0
    for i in range(1, len(prices)):
        diff = prices[i] - prices[i - 1]
        if diff > 0:
            profit += diff
    return profit

def can_complete_circuit(gas, cost):
    total = 0
    cur = 0
    start = 0
    for i in range(len(gas)):
        diff = gas[i] - cost[i]
        total += diff          # 全局可行性
        cur += diff
        if cur < 0:
            start = i + 1      # 起点移到下一站
            cur = 0
    return start if total >= 0 else -1

print('max_profit_ii([7,1,5,3,6,4]) =', max_profit_ii([7, 1, 5, 3, 6, 4]))  # 7
print('max_profit_ii([1,2,3,4,5]) =', max_profit_ii([1, 2, 3, 4, 5]))       # 4
print('can_complete_circuit([1,2,3,4,5],[3,4,5,1,2]) =',
      can_complete_circuit([1, 2, 3, 4, 5], [3, 4, 5, 1, 2]))   # 3
print('can_complete_circuit([2,3,4],[3,4,3]) =',
      can_complete_circuit([2, 3, 4], [3, 4, 3]))               # -1

## §5 分配类：分发饼干（455）/ 柠檬水找零（860）

### 455 分发饼干

**贪心**：小胃口吃小饼干——两个数组排序，双指针匹配：

$$\text{满足数} = \#\{i:\ \exists\ g_i \le s_j,\ \text{贪心最小匹配}\}$$

### 860 柠檬水找零

**贪心**：收到 20 时**优先找 10+5**（10 元更稀缺，先用掉）——局部最优保全局：

```python
if bill == 20:
    if tens > 0 and fives > 0: tens -= 1; fives -= 1
    elif fives >= 3: fives -= 3
    else: return False
```

**复杂度**：均 $O(n \log n)$（455 排序）/$O(n)$（860）。

In [ ]:
def find_content_children(g, s):
    g.sort(); s.sort()
    i = j = 0
    while i < len(g) and j < len(s):
        if s[j] >= g[i]:
            i += 1            # 满足一个孩子
        j += 1                # 饼干只用一次
    return i

def lemonade_change(bills):
    fives = tens = 0
    for b in bills:
        if b == 5:
            fives += 1
        elif b == 10:
            tens += 1
            fives -= 1
        else:                       # 20：优先 10+5
            if tens > 0 and fives > 0:
                tens -= 1
                fives -= 1
            elif fives >= 3:
                fives -= 3
            else:
                return False
        if fives < 0:
            return False
    return True

print('find_content_children:', find_content_children([1, 2, 3], [1, 1]))     # 1
print('find_content_children:', find_content_children([1, 2], [1, 2, 3]))     # 2
print('lemonade_change:', lemonade_change([5, 5, 5, 10, 20]))                  # True
print('lemonade_change:', lemonade_change([5, 5, 10, 10, 20]))                 # False（最后 20 找不开）

## §6 贪心证明框架（面试加分）

### 交换论证（Exchange Argument）

1. 假设一个最优解 $OPT$ 与贪心解 $G$ 第一次出现不同
2. 证明把 $OPT$ 的那个选择**换成** $G$ 的选择不会变差
3. 反复交换，$G$ 可转化为 $OPT$ → $G$ 也是最优

**区间调度举例**：$OPT$ 的第一个区间换成「结束最早」的区间，剩余空间更大，不可能更差。

### 贪心不成立时的识别信号

- 需要**全局比较**多个候选（硬币找零）→ DP
- 子问题会相互影响（背包）→ DP/搜索
- 题目有「最大化最小值 / 最小化最大值」→ 二分答案（主题 5）

## §7 常见 bug 与边界（面试踩坑区）

- **排序键选错**：区间题按左端点排是常见错误（435 必须按右端点；452 也是右端点）
- **边界相等**：435 的 `l >= end` 允许端点相接（不重叠）；452 的 `l > end` 才射新箭（端点相接可共箭）——两个不等号别搞混
- **`float('-inf')` 初始化**：end 初始用负无穷，否则第一个区间漏判
- **can_jump 判断顺序**：先判 `i > reach` 再更新 reach（顺序反了会误判）
- **jump 的边界**：`range(n - 1)` 不遍历最后一个（最后一步不用跳）
- **柠檬水找零**：先找 10+5 还是 3×5——顺序错了会局部失败
- **证明缺失**：贪心题只写代码不讲证明，面试官会追问「为什么贪心是对的」

## §8 面试追问与扩展（加分项）

### 8.1 贪心三问（写代码前自问）

1. 局部最优真的不损害全局吗？（举反例试）
2. 有没有需要比较多个方案的场景？（→ DP）
3. 能排序后获得单调性吗？（→ 排序 + 双指针/扫描）

### 8.2 高频变形

- 会议室 II（253）：区间重叠峰值 → 差分数组/最小堆
- 重构字符串（767）：优先放频次最高的字符
- 任务调度器（621）：冷却时间建模
- 贪心 + 堆：前 K 个高频（347，堆主题）

## §9 自测清单（L1 必会 / L2 进阶 / L3 挑战）

### L1 必会（10 题）

- [ ] 1. 买卖股票的最佳时机 II（LeetCode 122）
- [ ] 2. 分发饼干（455）
- [ ] 3. 柠檬水找零（860）
- [ ] 4. 跳跃游戏（55）
- [ ] 5. 跳跃游戏 II（45）
- [ ] 6. 最大子数组和（53，贪心/Kadane）
- [ ] 7. 分发糖果（135）
- [ ] 8. 最长回文串（409，计数贪心）
- [ ] 9. 数组拆分 I（561）
- [ ] 10. 种花问题（605）

### L2 进阶（10 题）

- [ ] 11. 无重叠区间（435）
- [ ] 12. 用最少数量的箭引爆气球（452）
- [ ] 13. 加油站（134）
- [ ] 14. 合并区间（56，排序贪心）
- [ ] 15. 划分字母区间（763）
- [ ] 16. 任务调度器（621）
- [ ] 17. 会议室 II（253）
- [ ] 18. 根据身高重建队列（406）
- [ ] 19. 单调递增的数字（738）
- [ ] 20. 摆动序列（376）

### L3 挑战（5 题）

- [ ] 21. 重构字符串（767）
- [ ] 22. 移掉 K 位数字（402，贪心+栈）
- [ ] 23. 分发糖果 II（困难变体）
- [ ] 24. 栅栏染色/粉刷房子（树形）
- [ ] 25. 砍竹子/切绳子的最大乘积（343，数学贪心）

### 自测要点

- 5 分钟内盲写：122 + 55 + 435
- 能给「区间调度按右端点排序」的交换论证
- 说出贪心 vs DP 的区分标准与一个反例

## 附录：参考与下节预告

- 《算法导论》第 16 章：贪心算法（活动选择是经典）
- LeetCode 分类题单：贪心

### 下节预告：主题 10 — 图（BFS/DFS/拓扑/并查集/最短路径/Trie）

> 💡 本主题验收：`高频题单.md`「贪心」一节全部打勾，能徒手证明 452 的贪心正确性。

In [ ]:
## §9.5 高频补充：分发糖果（135）/ 单调递增的数字（738）

def candy(ratings):
    """135 分发糖果：两次遍历，左→右（比左高则+1），右→左（比右高则取max）。"""
    n = len(ratings)
    left = [1] * n
    for i in range(1, n):
        if ratings[i] > ratings[i - 1]:
            left[i] = left[i - 1] + 1
    right = [1] * n
    for i in range(n - 2, -1, -1):
        if ratings[i] > ratings[i + 1]:
            right[i] = right[i + 1] + 1
    return sum(max(left[i], right[i]) for i in range(n))

def monotone_increasing_digits(n):
    """738 单调递增的数字：从右往左找第一个 s[i]>s[i+1]，该位-1、后面全变9。"""
    s = list(str(n))
    i = 0
    while i < len(s) - 1:
        if s[i] > s[i + 1]:
            break
        i += 1
    if i == len(s) - 1:
        return n                    # 本身已单调递增
    while i > 0 and s[i - 1] == s[i]:
        i -= 1                      # 连续相同位一起减（如 332 → 首位也减）
    s[i] = str(int(s[i]) - 1)
    for j in range(i + 1, len(s)):
        s[j] = '9'
    return int(''.join(s))

print('candy([1,0,2]) =', candy([1, 0, 2]), '（期望 5）')
print('candy([1,2,2]) =', candy([1, 2, 2]), '（期望 4）')
print('candy([1,2,87,87,87,2,1]) =', candy([1, 2, 87, 87, 87, 2, 1]), '（期望 13）')
print('monotone_increasing_digits(332) =', monotone_increasing_digits(332), '（期望 299）')
print('monotone_increasing_digits(1234) =', monotone_increasing_digits(1234), '（期望 1234）')
print('monotone_increasing_digits(10) =', monotone_increasing_digits(10), '（期望 9）')
